# 🐦‍⬛ RAVEN-1 — Post-Training on Colab

**Steps:**
1. Mount Drive & check GPU
2. Install dependencies
3. Upload code files
4. Copy posttrain.txt from Drive
5. Copy pre-trained weights from Drive
6. Pre-flight check
7. **Tokenize data** (`posttrain_data_prep.py`) — streaming, ~1 min for 1GB
8. **Post-train** (`posttrain.py`) — resume-safe
9. Test the model
10. Download weights

In [ ]:
# Step 1 — Mount Drive & check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
# Step 2 — Install dependencies
!pip install tokenizers numpy -q
print("✅ Dependencies installed")

---
## 📁 Step 3 — Upload Code Files

Upload these **6 files** to `/content/` using the Files panel (📁 on left):

| File | Purpose |
|------|--------|
| `config.py` | Hyperparameters & paths |
| `model.py` | Model architecture |
| `train.py` | Training utilities (shared) |
| `posttrain_data_prep.py` | Streaming tokenizer → binary files |
| `posttrain.py` | Post-training loop |
| `inference.py` | Chat/test interface |

In [ ]:
# Step 3 — Verify uploaded code files
import os

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("data", exist_ok=True)

required = ["config.py", "model.py", "train.py",
            "posttrain_data_prep.py", "posttrain.py"]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print("❌ Missing files! Upload these:")
    for f in missing: print(f"   → {f}")
else:
    print("✅ All code files found!")
    for f in required:
        print(f"   ✓ {f} ({os.path.getsize(f):,} bytes)")

In [ ]:
# Step 4 — Copy posttrain.txt from Drive
import shutil

# ╔════════════════════════════════════════════════════════════╗
# ║  👇 EDIT THIS PATH to your posttrain.txt on Drive         ║
# ╚════════════════════════════════════════════════════════════╝
DRIVE_POSTTRAIN = "/content/drive/MyDrive/posttrain.txt"

if os.path.exists("data/posttrain.txt"):
    size = os.path.getsize("data/posttrain.txt") / 1e6
    print(f"✅ data/posttrain.txt already exists ({size:.1f} MB)")
elif os.path.exists(DRIVE_POSTTRAIN):
    size_mb = os.path.getsize(DRIVE_POSTTRAIN) / 1e6
    print(f"📄 Found: {DRIVE_POSTTRAIN} ({size_mb:.1f} MB)")
    print("   Copying...")
    shutil.copy2(DRIVE_POSTTRAIN, "data/posttrain.txt")
    print(f"✅ Copied to data/posttrain.txt")
else:
    print(f"❌ Not found: {DRIVE_POSTTRAIN}")
    print("\n🔍 Searching Drive...")
    import subprocess
    result = subprocess.run(
        ["find", "/content/drive/MyDrive", "-name", "posttrain.txt", "-type", "f"],
        capture_output=True, text=True, timeout=30
    )
    if result.stdout.strip():
        for p in result.stdout.strip().split("\n"):
            s = os.path.getsize(p) / 1e6
            print(f"   📄 {p}  ({s:.1f} MB)")
        print("\n👆 Update DRIVE_POSTTRAIN above and re-run.")
    else:
        print("   Not found. Upload posttrain.txt to Drive first.")

In [ ]:
# Step 5 — Copy pre-trained weights from Drive
import shutil

# ╔════════════════════════════════════════════════════════════╗
# ║  👇 EDIT if your checkpoints are in a different folder     ║
# ╚════════════════════════════════════════════════════════════╝
DRIVE_CKPT = "/content/drive/MyDrive/raven1_checkpoints"

needed = {
    "raven1_best.pt": "Pre-trained weights (required)",
    "raven1_tokenizer.json": "Tokenizer (required)",
    "raven1_posttrain_latest.pt": "Resume checkpoint (optional)",
}

for filename, desc in needed.items():
    local = f"checkpoints/{filename}"
    drive_src = f"{DRIVE_CKPT}/{filename}"

    if os.path.exists(local):
        size = os.path.getsize(local) / 1e6
        print(f"  ✅ {filename} ({size:.1f} MB)")
    elif os.path.exists(drive_src):
        print(f"  📥 Copying {filename}...")
        shutil.copy2(drive_src, local)
        size = os.path.getsize(local) / 1e6
        print(f"  ✅ {filename} ({size:.1f} MB)")
    else:
        if "optional" in desc:
            print(f"  ⏭️  {filename} — {desc}, skipped")
        else:
            print(f"  ❌ {filename} — {desc}")

print()
ok = os.path.exists("checkpoints/raven1_best.pt") and os.path.exists("checkpoints/raven1_tokenizer.json")
print("✅ Weights ready!" if ok else "❌ Missing required files.")

In [ ]:
# Step 6 — Pre-flight check
checks = {
    "config.py": "Code",
    "model.py": "Code",
    "train.py": "Code",
    "posttrain_data_prep.py": "Code",
    "posttrain.py": "Code",
    "data/posttrain.txt": "Dataset",
    "checkpoints/raven1_best.pt": "Weights",
    "checkpoints/raven1_tokenizer.json": "Tokenizer",
}

all_ok = True
print("Pre-flight check:")
print("─" * 60)
for path, cat in checks.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        if size > 1e9: s = f"{size/1e9:.2f} GB"
        elif size > 1e6: s = f"{size/1e6:.1f} MB"
        else: s = f"{size/1e3:.1f} KB"
        print(f"  ✅ [{cat:>9s}] {path:<42s} {s}")
    else:
        print(f"  ❌ [{cat:>9s}] {path:<42s} MISSING")
        all_ok = False
print("─" * 60)
print("\n🚀 Ready! Run Step 7 → Step 8." if all_ok else "\n⛔ Fix missing files first.")

---
## 📊 Step 7 — Tokenize Post-Training Data

Streaming tokenizer — processes 1GB in ~1 minute without loading into RAM.

**Only runs once.** Subsequent runs auto-skip if data hasn't changed.

In [ ]:
# Step 7 — Tokenize (streaming, cached)
!python posttrain_data_prep.py

---
## 🚀 Step 8 — Run Post-Training

Resume-safe: if Colab disconnects, re-run Steps 1-5 then this cell with `--resume`.

Remove `--resume` for a fresh start.

In [ ]:
# Step 8 — Post-train (add --resume to continue from checkpoint)
!python posttrain.py --resume

---
## 🧪 Step 9 — Test the Model

In [ ]:
# Step 9 — Quick test
import torch
from tokenizers import Tokenizer
from model import Raven1
import config

model = Raven1().to(config.device)
ckpt_path = config.posttrain_weights_path
if not os.path.exists(ckpt_path):
    ckpt_path = config.weights_path
ckpt = torch.load(ckpt_path, map_location=config.device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded: {ckpt_path} | val_loss: {ckpt.get('best_val_loss', '?')}")

tokenizer = Tokenizer.from_file(config.tokenizer_path)
eos_id = tokenizer.token_to_id(config.eos_token)

prompts = ["hi", "what is gravity", "I failed my exam", "tell me a joke",
           "what is the meaning of life", "I'll start tomorrow"]

for p in prompts:
    text = f"{config.user_token} {p}\n{config.raven_token}"
    ids = torch.tensor([tokenizer.encode(text).ids], dtype=torch.long, device=config.device)
    out = model.generate(ids, max_new_tokens=80, temperature=0.65, top_k=25,
                         top_p=0.85, repetition_penalty=1.1, stop_token_id=eos_id)
    resp = tokenizer.decode(out[0, ids.size(1):].tolist()).strip()
    for tok in config.special_tokens:
        resp = resp.replace(tok, "")
    print(f"\n  User:  {p}")
    print(f"  Raven: {resp.strip() if resp.strip() else '(empty)'}")

---
## 💾 Step 10 — Download Weights

In [ ]:
# Step 10 — Download
from google.colab import files

if os.path.exists("checkpoints/raven1_posttrain_best.pt"):
    files.download("checkpoints/raven1_posttrain_best.pt")
    print("Downloaded raven1_posttrain_best.pt")
files.download("checkpoints/raven1_tokenizer.json")
print("\n✅ Also auto-synced to Drive: /content/drive/MyDrive/raven1_checkpoints/")